# 1.Importamos librerias

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

# 2. Estructura y carga de archivos

In [0]:
catalog = "smart_Claims"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
VOLUMEN_NAME = "landing"

smart_Claims = f"Volumes/{catalog}/{BRONZE_SCHEMA}/{VOLUMEN_NAME}/cutomers.csv"

smart_Claims = f"Volumes/{catalog}/{BRONZE_SCHEMA}/{VOLUMEN_NAME}/policies.csv"

smart_Claims = f"Volumes/{catalog}/{BRONZE_SCHEMA}/{VOLUMEN_NAME}/claims.csv"
spark = spark.builder.getOrCreate()

#creamos el catalogo
spark.sql(f"create catalog if not exists {catalog}")
#Creamos el esquema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{GOLD_SCHEMA}")
#Creamos el volumen
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{BRONZE_SCHEMA}.{VOLUMEN_NAME}")

##3. Construcción de la capa bronze
En esta capa se leen los archivos csv originales se normalizan los tipos básicos y se almacenan las tablas Delta.

In [0]:
#lee los customers
customers = (spark.read
.format("csv")
.option("header", True)
.option("inferSchema", True)
.csv("/Volumes/smart_Claims/bronze/landing/customers/customers.csv"))

#lee policies
policies = (spark.read.format("csv")
.option("header", True)
.option("inferSchema", True)
.csv("/Volumes/smart_Claims/bronze/landing/policies/policies.csv"))

#lee claims
claims = (spark.read.format("csv")
.option("header", True)
.option("inferSchema", True)
.csv("/Volumes/smart_Claims/bronze/landing/claims/claims.csv"))

#lee telematics
telematics = (spark.read.parquet("/Volumes/smart_Claims/bronze/landing/telematics"))

#lee claim_images
claim_images = (spark.read.format("binaryFile").load("/Volumes/smart_Claims/bronze/landing/claim_images"))

#lee claim_metadata
claim_metadata = (spark.read.format("csv")
.option("header", True)
.option("inferSchema", True)
.load("/Volumes/smart_Claims/bronze/landing/claim_metadata"))

### 3.1 Validación de datos

In [0]:
#validación
print(customers.count())
print(policies.count())
print(claims.count())
print(telematics.count())
print(claim_images.count())
print(claim_metadata.count())

In [0]:
display(customers.limit(7))

In [0]:
# Guardar as tablas Bronze

claims.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{BRONZE_SCHEMA}.claims")

policies.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{BRONZE_SCHEMA}.policies")

customers.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{BRONZE_SCHEMA}.customers")

telematics.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{BRONZE_SCHEMA}.telematics")

claim_images.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{BRONZE_SCHEMA}.claim_images")

claim_metadata.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{BRONZE_SCHEMA}.claim_metadata")